# W8-3 Stockout risk, reorder points and back-test (180 stock pairs)
**Owner:** Ibrahim Khalil Umar · **Week 8** · reviewer: Abdibasid Mohamed Ali

**Questions.** (1) Which product-branch pairs are likely to run out? (2) What reorder point should each pair use, and how did it do in a back-test? The Week 7 lead-time-aware trigger is extended: parameters are fitted on the **first four years (2019 to 2022) only** and judged on the **last two (2023 to 2025)**.

> **Every stock figure here rests on one assumption: that the sales scale is the right reference for stock (Q10 unresolved).** No holding or ordering cost data exists, so no policy is called cost-optimal.

In [1]:
import os, pathlib, math, time
import numpy as np, pandas as pd
from scipy.stats import norm, spearmanr, rankdata
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 42; RESAMPLES = 5000; REPS = 500; WARM = 60
TRAIN_END = pd.Timestamp("2022-12-31")
HERE = pathlib.Path.cwd()
def find_merged():
    env = os.environ.get("CADETX_MERGED")
    if env and pathlib.Path(env).exists():
        return pathlib.Path(env)
    for p in [HERE, *HERE.parents]:
        c = p / "week-02" / "reports" / "week_02" / "merged"
        if c.exists():
            return c
    raise SystemExit("Merged datasets not found: set CADETX_MERGED or run inside the repository checkout.")
MERGED = find_merged()
OUT = HERE / "w83_outputs"; CH = HERE / "w83_charts"; OUT.mkdir(exist_ok=True); CH.mkdir(exist_ok=True)
LABEL = "Sales scale assumed (Q10 unresolved); results rest on this assumption"
ASSUMPTIONS = ("parameters fitted on 2019-2022 only, judged on 2023-2025; lost sales, no backorder; each policy starts full (order-up-to level) "
               "after a 60-day warm-up; order lead times drawn from the 2023-2025 received purchase-order lines of the pair; "
               "no holding or ordering cost, so no policy is called cost-optimal")
GREY, INK, INK2, GRID, SURF = "#8A8F98", "#16213A", "#4A5163", "#E3E5EA", "#FFFFFF"
C_A, C_B, C_BLT = "#C8791A", "#1B6B66", "#16213A"
checks = []
def check(name, expected, actual, ok):
    checks.append(dict(Check_Item=name, Expected=expected, Actual=actual, Status="PASS" if ok else "FAIL"))
    assert ok, f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top", "right"]: ax.spines[s].set_visible(False)
    for s in ["left", "bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True, color=GRID, lw=.8); ax.set_axisbelow(True)

K = ["product_id", "branch_id"]
L = pd.read_csv(MERGED / "stock_ledger_enriched.csv", low_memory=False, parse_dates=["movement_date"])
I = pd.read_csv(MERGED / "inventory_with_ledger.csv")
P = pd.read_csv(MERGED / "purchase_orders_full.csv", low_memory=False, parse_dates=["order_date", "expected_delivery_date", "received_date"],
                usecols=["po_id", "product_id", "branch_id", "supplier_id", "po_status", "order_date", "expected_delivery_date", "received_date",
                         "supplier_lead_days", "lead_time_actual_days", "delivery_delay_days"])
print("data folder:", MERGED)
tot_led = L.groupby("movement_type").quantity.sum()
check("ledger IN is about 18.1 times OUT (Q10 reproduces)", "18.13 +/- 0.1", round(tot_led["IN"] / tot_led["OUT"], 2), abs(tot_led["IN"] / tot_led["OUT"] - 18.13) < 0.1)
check("180 product-branch pairs", "180", str(len(I)), len(I) == 180)

data folder: /home/claude/final/merged


## 1. Demand, train/test split and lead-time history

In [2]:
O = L[L.movement_type == "OUT"]
dates = pd.date_range(O.movement_date.min(), O.movement_date.max(), freq="D")
pidx = pd.MultiIndex.from_frame(I[K]); NP = len(I)
D = O.groupby(K + ["movement_date"]).quantity.sum().unstack("movement_date").reindex(columns=dates, fill_value=0).fillna(0).reindex(pidx, fill_value=0).values.astype(float)
T = D.shape[1]; t_cut = int((dates <= TRAIN_END).sum())
check("demand matrix total equals ledger OUT total", str(int(O.quantity.sum())), str(int(D.sum())), int(D.sum()) == int(O.quantity.sum()))
check("full daily calendar of 2,203 days", "(180, 2203)", str(D.shape), D.shape == (180, 2203))
check("train window is 2019 to 2022 and the test window holds at least two years", ">= 730 test days", f"train {t_cut}, test {T - t_cut}", t_cut + (T - t_cut) == T and T - t_cut >= 730)

R = P[P.po_status.eq("Received") & P.lead_time_actual_days.notna()].copy()
check("lead_time_actual_days equals received date minus order date for every received line", "100%", f"{((R.received_date - R.order_date).dt.days == R.lead_time_actual_days).mean():.4f}", ((R.received_date - R.order_date).dt.days == R.lead_time_actual_days).all())
check("supplier_lead_days equals expected delivery minus order date for every received line", "100%", f"{((R.expected_delivery_date - R.order_date).dt.days == R.supplier_lead_days).mean():.4f}", ((R.expected_delivery_date - R.order_date).dt.days == R.supplier_lead_days).all())
R_tr = R[R.order_date <= TRAIN_END]; R_te = R[R.order_date > TRAIN_END]
def emp_matrix(Rx):
    g = Rx.groupby(K).lead_time_actual_days
    e = [g.get_group(tuple(k)).values.astype(int) for k in pidx]
    c = np.array([len(x) for x in e]); M = np.zeros((NP, c.max()), int)
    for i, x in enumerate(e): M[i, :len(x)] = x
    return e, c, M
emp_tr, cnt_tr, E_tr = emp_matrix(R_tr); emp_te, cnt_te, E_te = emp_matrix(R_te)
check("every pair has at least 400 training and 200 test received lines", ">= 400 / >= 200", f"{cnt_tr.min()} / {cnt_te.min()}", cnt_tr.min() >= 400 and cnt_te.min() >= 200)
ltm = np.array([x.mean() for x in emp_tr]); lts = np.array([x.std(ddof=1) for x in emp_tr])     # training lead-time stats
mu = D[:, :t_cut].mean(axis=1); sd = D[:, :t_cut].std(axis=1, ddof=1)                          # training demand stats
mu_te = D[:, t_cut:].mean(axis=1)
stab = pd.DataFrame({"measure": ["median pair: test mean daily demand / train mean daily demand", "median pair: test mean lead time minus train mean lead time (days)",
                                 "pairs whose test demand is within 15% of train demand"],
                     "value": [np.median(mu_te / mu), np.median(np.array([x.mean() for x in emp_te]) - ltm), int((np.abs(mu_te / mu - 1) <= .15).sum())]})
stab["label"] = LABEL; stab.to_csv(OUT / "w83_train_test_stability.csv", index=False); print(stab.round(3).to_string(index=False))
check("training demand uses only 2019 to 2022 days", "mean over first %d days" % t_cut, "ok", np.allclose(mu, D[:, :t_cut].mean(axis=1)) and not np.allclose(mu, D.mean(axis=1)))

# lead time by supplier: where the variation comes from
sup = R.groupby("supplier_id").agg(received_lines=("po_id", "size"), promised_lead_days=("supplier_lead_days", "first"),
        actual_lead_mean=("lead_time_actual_days", "mean"), actual_lead_sd=("lead_time_actual_days", "std"),
        share_late=("delivery_delay_days", lambda x: (x > 0).mean()), share_early=("delivery_delay_days", lambda x: (x < 0).mean()),
        delay_min=("delivery_delay_days", "min"), delay_max=("delivery_delay_days", "max")).reset_index()
within = (R.groupby("supplier_id").lead_time_actual_days.var() * R.groupby("supplier_id").size()).sum() / len(R)
share_between = 1 - within / R.lead_time_actual_days.var()
sup["label"] = LABEL; sup.round(3).to_csv(OUT / "w83_lead_time_by_supplier.csv", index=False)
print(sup.round(3).drop(columns="label").to_string(index=False)); print(f"share of lead-time variance between suppliers: {share_between:.3f}")
check("every pair is supplied by all 8 suppliers", "8 per pair", str(int(R.groupby(K).supplier_id.nunique().min())) + "-" + str(int(R.groupby(K).supplier_id.nunique().max())), R.groupby(K).supplier_id.nunique().eq(8).all())
check("more than 90% of lead-time variance is between suppliers (the supplier mix, not delay, drives the spread)", "> 0.90", round(share_between, 3), share_between > .90)
check("delay against the promised date runs from -3 to +7 days for every supplier", "-3 to +7", f"{int(sup.delay_min.min())} to {int(sup.delay_max.max())}", sup.delay_min.min() == -3 and sup.delay_max.max() == 7)

                                                           measure   value                                                                 label
     median pair: test mean daily demand / train mean daily demand   0.970 Sales scale assumed (Q10 unresolved); results rest on this assumption
median pair: test mean lead time minus train mean lead time (days)  -0.151 Sales scale assumed (Q10 unresolved); results rest on this assumption
             pairs whose test demand is within 15% of train demand 159.000 Sales scale assumed (Q10 unresolved); results rest on this assumption
supplier_id  received_lines  promised_lead_days  actual_lead_mean  actual_lead_sd  share_late  share_early  delay_min  delay_max
    SUP0001           17490                  19            19.479           1.678       0.147        0.045       -3.0        7.0
    SUP0002           17227                  10            10.511           1.722       0.150        0.042       -3.0        7.0
    SUP0003           17450      

## 2. Simulator with a counting window
Same rules as Weeks 5 to 7 (daily review; order up to `S` when on hand plus on order is at or below `s`; lost sales). Each policy starts full and runs through a 60-day warm-up on the last days of the training period; only the test period is counted. Order lead times are drawn from the pair's received purchase-order lines.

In [3]:
def simulate(s, S, reps, seed, t_start, t_end, t_count, Em, cn_pair, U=None, pairs=None):
    """Vectorised over pairs x reps. Counts only days >= t_count. Starts full (on hand = S) at t_start."""
    pairs = np.arange(NP) if pairs is None else np.asarray(pairs)
    n = len(pairs); N = n * reps; rng = np.random.default_rng(seed)
    s_ = np.tile(s, reps); S_ = np.tile(S, reps); pr = np.tile(pairs, reps)
    B = 64; pipe = np.zeros((N, B)); onhand = S_.copy(); onorder = np.zeros(N)
    unmet = np.zeros(N); episodes = np.zeros(N); sdays = np.zeros(N); area = np.zeros(N); orders = np.zeros(N); prev = np.zeros(N, bool)
    rows = np.arange(N); cn = cn_pair[pr]; days = t_end - t_count
    for t in range(t_start, t_end):
        sl = t % B; a = pipe[:, sl].copy(); pipe[:, sl] = 0; onhand += a; onorder -= a
        d = np.tile(D[pairs, t], reps); sold = np.minimum(d, onhand); short = d > sold; onhand -= sold
        pos = onhand + onorder; q = S_ - pos; m = (pos <= s_) & (q > 0)
        u = U[:, t - t_start] if U is not None else rng.random(N)
        if t >= t_count:
            unmet += d - sold; sdays += short; episodes += (short & ~prev); area += onhand
        prev = short
        if m.any():
            r = rows[m]; lead = Em[pr[r], np.minimum((u[r] * cn[r]).astype(int), cn[r] - 1)]
            pipe[r, (t + lead) % B] += q[r]; onorder[r] += q[r]
            if t >= t_count: orders[r] += 1
    f = lambda x: x.reshape(reps, n)
    return dict(unmet=f(unmet), episodes=f(episodes), stockout_days=f(sdays), avg_on_hand=f(area / days), orders=f(orders))

def simulate_scalar(i, s, S, U_row, t_start, t_end, t_count, Em, cn_pair):
    pipe = {}; onhand = float(S); onorder = 0.0; unmet = 0.0; area = 0.0; orders = 0; episodes = 0; prev = False
    for t in range(t_start, t_end):
        a = pipe.pop(t, 0.0); onhand += a; onorder -= a
        d = D[i, t]; sold = min(d, onhand); short = d > sold; onhand -= sold
        pos = onhand + onorder
        if t >= t_count:
            unmet += d - sold; area += onhand; episodes += int(short and not prev)
        prev = short
        if pos <= s and S - pos > 0:
            lead = Em[i, min(int(U_row[t - t_start] * cn_pair[i]), cn_pair[i] - 1)]
            pipe[t + lead] = pipe.get(t + lead, 0.0) + (S - pos); onorder += S - pos
            if t >= t_count: orders += 1
    return unmet, area / (t_end - t_count), orders, episodes

sA = I.reorder_level.values.astype(float); SA = I.max_stock.values.astype(float); gap = SA - sA
def s_mean(z): return np.ceil(mu * ltm + z * sd * np.sqrt(ltm))                                    # mean-lead trigger (Week 6 form)
def s_lt(z):   return np.ceil(mu * ltm + z * np.sqrt(ltm * sd**2 + mu**2 * lts**2))               # lead-time-aware trigger (Week 7)
POL = {"A existing (reorder_level, max_stock)": (sA, SA)}
s = s_mean(1.645); POL["B 95% (mean-lead trigger)"] = (s, s + gap)
for z, nm in [(1.28, "90%"), (1.645, "95%"), (2.33, "99%")]:
    s = s_lt(z); POL[f"B-LT {nm} (lead-time-aware trigger)"] = (s, s + gap)
t_start = t_cut - WARM

# cross-check on 5 pairs with identical random draws
chk = [0, 37, 71, 120, 179]; U = np.random.default_rng(SEED).random((5, T - t_start))
for nm in ["A existing (reorder_level, max_stock)", "B-LT 95% (lead-time-aware trigger)"]:
    s_, S_ = POL[nm]
    v = simulate(s_[chk], S_[chk], 1, SEED, t_start, T, t_cut, E_te, cnt_te, U=U, pairs=chk)
    ok = True
    for k, i in enumerate(chk):
        u_, av_, o_, ep_ = simulate_scalar(i, s_[i], S_[i], U[k], t_start, T, t_cut, E_te, cnt_te)
        ok &= abs(u_ - v["unmet"][0, k]) < 1e-6 and abs(av_ - v["avg_on_hand"][0, k]) < 1e-6 and o_ == v["orders"][0, k] and ep_ == v["episodes"][0, k]
    check(f"vectorised simulator equals a plain day-by-day loop on 5 pairs ({nm.split(' (')[0]})", "identical", "identical" if ok else "differs", ok)

## 3. Back-test: fitted on 2019 to 2022, judged on 2023 to 2025
500 lead-time paths (seed 42), the same draws for every policy.

In [4]:
test_dem = D[:, t_cut:].sum(axis=1); tot_dem = test_dem.sum(); cost = I.unit_cost.values.astype(float)
RES = {}; rows = []; PAIR = []
t0 = time.time()
for name, (s_, S_) in POL.items():
    r = simulate(s_, S_, REPS, SEED, t_start, T, t_cut, E_te, cnt_te); RES[name] = r
    fr = 1 - r["unmet"].sum(1) / tot_dem; pf = 1 - r["unmet"] / test_dem; val = (r["avg_on_hand"] * cost).sum(1)
    rows.append(dict(policy=name, replications=REPS, fill_rate=fr.mean(), fill_p2_5=np.percentile(fr, 2.5), fill_p97_5=np.percentile(fr, 97.5),
                     pairs_below_95=int((pf.mean(0) < .95).sum()), stockout_episodes_per_pair_per_year=r["episodes"].mean() / ((T - t_cut) / 365.25),
                     stockout_day_share=r["stockout_days"].sum(1).mean() / (NP * (T - t_cut)),
                     avg_stock_value_at_cost=val.mean(), value_p2_5=np.percentile(val, 2.5), value_p97_5=np.percentile(val, 97.5), orders_placed=r["orders"].sum(1).mean()))
print("simulated in", round(time.time() - t0, 1), "s")
SUM = pd.DataFrame(rows); SUM["label"] = LABEL; SUM["assumptions"] = ASSUMPTIONS; SUM.to_csv(OUT / "w83_backtest_summary.csv", index=False)
print(SUM.drop(columns=["label", "assumptions", "replications"]).round(4).to_string(index=False))
A = "A existing (reorder_level, max_stock)"
for n, r in RES.items():
    pass
check("fill rate lies between 0 and 1 for every policy and pair", "0 to 1", "ok", all(((1 - r["unmet"] / test_dem) >= -1e-9).all() and ((1 - r["unmet"] / test_dem) <= 1 + 1e-9).all() for r in RES.values()))
check("500 replications, seed 42, 5,000 bootstrap resamples", "500 / 42 / 5000", f"{REPS} / {SEED} / {RESAMPLES}", REPS == 500 and SEED == 42 and RESAMPLES == 5000)
sm = SUM.set_index("policy")
check("higher service factor never lowers back-test fill (B-LT 90 < 95 < 99)", "monotone", "monotone",
      sm.loc["B-LT 90% (lead-time-aware trigger)", "fill_rate"] < sm.loc["B-LT 95% (lead-time-aware trigger)", "fill_rate"] < sm.loc["B-LT 99% (lead-time-aware trigger)", "fill_rate"])
check("policies were counted over the same test window (744 days) after a 60-day warm-up", "T - t_cut days", str(T - t_cut), T - t_cut == 744 and t_start == t_cut - 60)

simulated in 11.1 s
                               policy  fill_rate  fill_p2_5  fill_p97_5  pairs_below_95  stockout_episodes_per_pair_per_year  stockout_day_share  avg_stock_value_at_cost   value_p2_5  value_p97_5  orders_placed
A existing (reorder_level, max_stock)     0.9384     0.9348      0.9419              78                               4.2589              0.0144             3.971404e+08 3.927617e+08 4.023438e+08       1473.766
            B 95% (mean-lead trigger)     0.9891     0.9876      0.9907               0                               0.7968              0.0026             4.744104e+08 4.696741e+08 4.790578e+08       1590.374
   B-LT 90% (lead-time-aware trigger)     0.9876     0.9859      0.9893               2                               0.9219              0.0030             4.666600e+08 4.617625e+08 4.715258e+08       1587.014
   B-LT 95% (lead-time-aware trigger)     0.9929     0.9914      0.9941               0                               0.5449            

## 4. Does the fitted lead-time-aware trigger beat the existing levels out of sample?
Paired differences against A under the same draws; the pair-level interval is a bootstrap over the 180 pairs (5,000 resamples, seed 42) of each pair's mean fill difference.

In [5]:
rng = np.random.default_rng(SEED); bi = rng.integers(0, NP, size=(RESAMPLES, NP))
pfA = (1 - RES[A]["unmet"] / test_dem).mean(0); frA = 1 - RES[A]["unmet"].sum(1) / tot_dem; valA = (RES[A]["avg_on_hand"] * cost).sum(1)
rows = []
for n in [x for x in POL if x != A]:
    pf = (1 - RES[n]["unmet"] / test_dem).mean(0); fr = 1 - RES[n]["unmet"].sum(1) / tot_dem; dpair = pf - pfA; boot = dpair[bi].mean(1)
    val = (RES[n]["avg_on_hand"] * cost).sum(1)
    rows.append(dict(policy=n, fill_gain_pts=100 * (fr - frA).mean(), gain_p2_5=100 * np.percentile(fr - frA, 2.5), gain_p97_5=100 * np.percentile(fr - frA, 97.5),
                     share_of_paths_beats_A=float((fr > frA).mean()), mean_pair_gain_pts=100 * dpair.mean(), pair_ci_low=100 * np.percentile(boot, 2.5), pair_ci_high=100 * np.percentile(boot, 97.5),
                     pairs_higher=int((dpair > 0).sum()), pairs_lower=int((dpair < 0).sum()), extra_stock_value_pct=100 * (val / valA - 1).mean()))
CMP = pd.DataFrame(rows); CMP["label"] = LABEL; CMP["assumptions"] = ASSUMPTIONS; CMP.to_csv(OUT / "w83_comparison_vs_A.csv", index=False)
print(CMP.drop(columns=["label", "assumptions"]).round(3).to_string(index=False))
lt95 = "B-LT 95% (lead-time-aware trigger)"
check("fitted B-LT 95% beats A in 100% of out-of-sample paths", "1.0", str(float(CMP.set_index("policy").loc[lt95, "share_of_paths_beats_A"])), CMP.set_index("policy").loc[lt95, "share_of_paths_beats_A"] == 1.0)
check("pair-level bootstrap interval for the B-LT 95% fill gain excludes zero", "low > 0", str(round(CMP.set_index("policy").loc[lt95, "pair_ci_low"], 2)), CMP.set_index("policy").loc[lt95, "pair_ci_low"] > 0)

                            policy  fill_gain_pts  gain_p2_5  gain_p97_5  share_of_paths_beats_A  mean_pair_gain_pts  pair_ci_low  pair_ci_high  pairs_higher  pairs_lower  extra_stock_value_pct
         B 95% (mean-lead trigger)          5.078      4.711       5.436                     1.0               4.874        4.068         5.695           159           19                 19.458
B-LT 90% (lead-time-aware trigger)          4.923      4.551       5.280                     1.0               4.712        3.911         5.526           154           24                 17.507
B-LT 95% (lead-time-aware trigger)          5.450      5.092       5.789                     1.0               5.234        4.404         6.079           166            9                 25.592
B-LT 99% (lead-time-aware trigger)          5.931      5.597       6.298                     1.0               5.709        4.850         6.592           176            1                 41.144


## 5. Stockout-risk prediction: can a score built on 2019 to 2022 pick out the pairs that run short in 2023 to 2025?
The score uses training data only: the service factor the existing reorder level implies, `z = (s - mean lead-time demand) / SD of lead-time demand`, turned into a cycle stockout probability `1 - Phi(z)` (normal approximation, ignores `max_stock`). The outcome is whether the pair's back-test fill under policy A is below 95%. Two simple baselines are compared.

In [6]:
ltd = mu * ltm; rho_ltd, p_ltd = spearmanr(sA, ltd)
stab = pd.concat([stab, pd.DataFrame({"measure": ["existing reorder_level: coefficient of variation across pairs", "training mean lead-time demand: coefficient of variation across pairs",
        "Spearman, existing reorder_level vs training mean lead-time demand", "  p-value of that correlation"],
        "value": [sA.std() / sA.mean(), ltd.std() / ltd.mean(), rho_ltd, p_ltd], "label": LABEL})], ignore_index=True)
stab.to_csv(OUT / "w83_train_test_stability.csv", index=False)
check("existing reorder levels are not related to the lead-time demand they should cover (|Spearman| < 0.3)", "< 0.3", round(rho_ltd, 3), abs(rho_ltd) < .3)
Lbar = ltm; sdLTD = np.sqrt(ltm * sd**2 + mu**2 * lts**2)
zA = (sA - mu * ltm) / sdLTD; p_cycle = 1 - norm.cdf(zA)
pfA_te = pfA; at_risk = pfA_te < .95
shortfall = 1 - pfA_te
# training-period persistence baseline: A simulated on 2019-2022 with training lead times
rtr = simulate(sA, SA, 100, SEED, 0, t_cut, WARM, E_tr, cnt_tr); pf_train = (1 - rtr["unmet"] / D[:, WARM:t_cut].sum(1)).mean(0)
def auc(score, y):
    r = rankdata(score); n1 = y.sum(); n0 = len(y) - n1
    return (r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
def auc_brute(score, y):
    pos = score[y]; neg = score[~y]; return ((pos[:, None] > neg[None, :]).sum() + .5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg))
scores = {"Model: cycle stockout probability (training data)": p_cycle, "Baseline 1: low reorder_level only": -sA,
          "Baseline 2: low cover ratio (reorder_level / mean lead-time demand)": -(sA / (mu * ltm)), "Baseline 3: training-period fill under A (persistence)": 1 - pf_train}
rows = []; rng = np.random.default_rng(SEED)
bi2 = rng.integers(0, NP, size=(RESAMPLES, NP))
for nm, sc in scores.items():
    a = auc(sc, at_risk); bs = []
    for b in bi2:
        yb = at_risk[b]
        if yb.sum() in (0, NP): continue
        bs.append(auc(sc[b], yb))
    rho, _ = spearmanr(sc, shortfall)
    perm = np.random.default_rng(SEED); pm = np.array([abs(spearmanr(sc, perm.permutation(shortfall))[0]) for _ in range(RESAMPLES)])
    top = np.argsort(-sc)[:20]
    rows.append(dict(score=nm, auc=a, auc_ci_low=np.percentile(bs, 2.5), auc_ci_high=np.percentile(bs, 97.5), spearman_with_shortfall=rho,
                     permutation_p=(1 + (pm >= abs(rho)).sum()) / (RESAMPLES + 1), top20_truly_below_95=int(at_risk[top].sum())))
RISK = pd.DataFrame(rows); RISK["pairs_below_95_in_test"] = int(at_risk.sum()); RISK["label"] = LABEL; RISK.to_csv(OUT / "w83_risk_validation.csv", index=False)
print(f"pairs below 95% fill under A in the test period: {int(at_risk.sum())} of 180"); print(RISK.drop(columns="label").round(3).to_string(index=False))
check("AUC by ranks equals AUC by brute-force pair comparison", "equal", "equal", all(abs(auc(sc, at_risk) - auc_brute(sc, at_risk)) < 1e-9 for sc in scores.values()))
check("cycle stockout probabilities lie between 0 and 1", "0 to 1", "ok", ((p_cycle >= 0) & (p_cycle <= 1)).all())
check("test period has both at-risk and adequate pairs under A", "mixed", f"{int(at_risk.sum())} at risk", 0 < at_risk.sum() < NP)

pairs below 95% fill under A in the test period: 78 of 180
                                                              score   auc  auc_ci_low  auc_ci_high  spearman_with_shortfall  permutation_p  top20_truly_below_95  pairs_below_95_in_test
                  Model: cycle stockout probability (training data) 0.966       0.941        0.986                    0.923            0.0                    20                      78
                                 Baseline 1: low reorder_level only 0.965       0.938        0.986                    0.909            0.0                    20                      78
Baseline 2: low cover ratio (reorder_level / mean lead-time demand) 0.965       0.939        0.985                    0.918            0.0                    20                      78
             Baseline 3: training-period fill under A (persistence) 0.971       0.950        0.988                    0.933            0.0                    20                      78


## 6. Reorder point and back-test result for every pair

In [7]:
PAIRRES = pd.DataFrame({"product_id": I.product_id, "branch_id": I.branch_id, "category": I.category,
    "train_mean_daily_demand": mu.round(3), "train_sd_daily_demand": sd.round(3), "train_lead_mean_days": ltm.round(2), "train_lead_sd_days": lts.round(2),
    "existing_reorder_level": sA, "existing_max_stock": SA,
    "reorder_point_90": s_lt(1.28), "reorder_point_95": s_lt(1.645), "reorder_point_99": s_lt(2.33),
    "order_up_to_95": s_lt(1.645) + gap,
    "reorder_point_95_days_of_demand": (s_lt(1.645) / mu).round(1),
    "expected_days_between_orders_95": ((gap) / mu).round(1),
    "implied_service_factor_of_existing": zA.round(3), "predicted_cycle_stockout_prob_existing": p_cycle.round(4),
    "backtest_fill_existing": pfA_te.round(4),
    "backtest_fill_recommended_95": (1 - RES[lt95]["unmet"] / test_dem).mean(0).round(4),
    "backtest_stockout_episodes_per_year_existing": (RES[A]["episodes"].mean(0) / ((T - t_cut) / 365.25)).round(2),
    "backtest_stockout_episodes_per_year_recommended_95": (RES[lt95]["episodes"].mean(0) / ((T - t_cut) / 365.25)).round(2),
    "existing_below_95_in_backtest": at_risk,
    "recommended_vs_existing_reorder_point": np.where(s_lt(1.645) > sA, "raise", "lower or keep")})
PAIRRES["label"] = LABEL; PAIRRES["assumptions"] = ASSUMPTIONS; PAIRRES.to_csv(OUT / "w83_pair_reorder_backtest.csv", index=False)
print(PAIRRES[["product_id", "branch_id", "existing_reorder_level", "reorder_point_95", "backtest_fill_existing", "backtest_fill_recommended_95"]].head(8).to_string(index=False))
catt = PAIRRES.groupby("category").agg(pairs=("product_id", "size"), existing_below_95=("existing_below_95_in_backtest", "sum"), raise_reorder_point=("recommended_vs_existing_reorder_point", lambda x: int((x == "raise").sum()))).reset_index()
catt["label"] = LABEL; catt.to_csv(OUT / "w83_by_category.csv", index=False); print(catt.to_string(index=False))
check("every pair has a reorder point of at least 1 unit and an order-up-to level above it", "all 180", "all 180", (PAIRRES.reorder_point_95 >= 1).all() and (PAIRRES.order_up_to_95 > PAIRRES.reorder_point_95).all())
check("one row per pair in the reorder table", "180", str(len(PAIRRES)), len(PAIRRES) == 180 and not PAIRRES.duplicated(K).any())
check("reorder points rise with the service level for every pair (90 <= 95 <= 99)", "all pairs", "all pairs", ((PAIRRES.reorder_point_90 <= PAIRRES.reorder_point_95) & (PAIRRES.reorder_point_95 <= PAIRRES.reorder_point_99)).all())

product_id branch_id  existing_reorder_level  reorder_point_95  backtest_fill_existing  backtest_fill_recommended_95
      P001    DEL001                    59.0             101.0                  0.9764                        0.9934
      P001    PUN001                    70.0              87.0                  0.9859                        0.9949
      P001    CHN001                    62.0             135.0                  0.9245                        0.9928
      P001    HYD001                    71.0              90.0                  0.9990                        0.9999
      P001    KOL001                    94.0             118.0                  0.9904                        0.9975
      P001    AHM001                    77.0             114.0                  0.9801                        0.9960
      P002    DEL001                    57.0              97.0                  0.9417                        0.9928
      P002    PUN001                    41.0              99.0  

## 7. Charts

In [8]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.8))
r_ = RISK.iloc[0]
ax[0].scatter(p_cycle * 100, pfA_te * 100, s=16, c=np.where(at_risk, C_A, C_B), alpha=.8)
ax[0].axhline(95, color=GREY, ls="--", lw=.9); ax[0].set_xlabel("Predicted cycle stockout probability from 2019-2022 data, %"); ax[0].set_ylabel("Back-test fill rate under existing levels, 2023-2025, %")
ax[0].set_title(f"Training-data risk score vs what happened (AUC {r_.auc:.2f})"); style(ax[0])
names = list(POL); short = ["A existing", "B 95% mean-lead", "B-LT 90%", "B-LT 95%", "B-LT 99%"]; cols = [C_A, C_B, C_BLT, C_BLT, C_BLT]
vv = [sm.loc[n, "fill_rate"] * 100 for n in names]; lo = [sm.loc[n, "fill_p2_5"] * 100 for n in names]; hi = [sm.loc[n, "fill_p97_5"] * 100 for n in names]
ax[1].bar(range(5), vv, color=cols); ax[1].errorbar(range(5), vv, yerr=[np.array(vv) - np.array(lo), np.array(hi) - np.array(vv)], fmt="none", ecolor=INK, capsize=3, lw=1)
for i, v_ in enumerate(vv): ax[1].text(i, hi[i] + .12, f"{v_:.1f}", ha="center", fontsize=9, fontweight="bold")
ax[1].set_xticks(range(5)); ax[1].set_xticklabels(short, fontsize=9); ax[1].set_ylim(90, 101); ax[1].set_ylabel("Back-test fill rate, %"); ax[1].set_title("Fitted on 2019-2022, judged on 2023-2025"); style(ax[1])
fig.text(.5, -.02, LABEL, ha="center", fontsize=8.5, color=INK2); plt.tight_layout(); plt.savefig(CH / "w83_01_risk_and_backtest_fill.png", dpi=150, facecolor=SURF, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(7.6, 5.4))
ax.scatter(sA, PAIRRES.reorder_point_95, s=16, c=np.where(at_risk, C_A, C_B), alpha=.8)
m_ = max(sA.max(), PAIRRES.reorder_point_95.max()) * 1.05; ax.plot([0, m_], [0, m_], color=GREY, ls="--", lw=.9)
ax.set_xlabel("Existing reorder_level, units"); ax.set_ylabel("Recommended reorder point (95%, lead-time-aware), units"); ax.set_title("The recommended point is above the existing level for almost every pair")
ax.text(.03, .94, "Orange: below 95% fill in the back-test under existing levels", transform=ax.transAxes, fontsize=9, color=C_A); style(ax)
fig.text(.5, -.02, LABEL, ha="center", fontsize=8.5, color=INK2); plt.tight_layout(); plt.savefig(CH / "w83_02_reorder_point_vs_existing.png", dpi=150, facecolor=SURF, bbox_inches="tight"); plt.close()

fig, ax = plt.subplots(figsize=(8.8, 5)); 
for n, sh, c, mk in zip(names, short, cols, ["o", "s", "D", "D", "D"]):
    x_ = sm.loc[n, "avg_stock_value_at_cost"] / 1e6; y_ = sm.loc[n, "fill_rate"] * 100
    ax.scatter(x_, y_, s=70, color=c, marker=mk, zorder=3); ax.annotate(sh, (x_, y_), textcoords="offset points", xytext=(7, -4), fontsize=9, color=INK2)
ax.set_xlabel("Average stock value at cost, Rs million"); ax.set_ylabel("Back-test fill rate, %"); ax.set_title("More fill needs more stock (2023-2025 back-test)"); style(ax)
fig.text(.5, -.02, "No holding or ordering cost data: stock value is a proxy, no point is called cost-optimal | " + LABEL, ha="center", fontsize=8.5, color=INK2)
plt.tight_layout(); plt.savefig(CH / "w83_03_fill_vs_stock_value.png", dpi=150, facecolor=SURF, bbox_inches="tight"); plt.close()
check("three charts written", "3", str(len(list(CH.glob("*.png")))), len(list(CH.glob("*.png"))) == 3)

## 8. Validation

In [9]:
outs = sorted(p.name for p in OUT.glob("*.csv"))
need = ["w83_backtest_summary.csv", "w83_pair_reorder_backtest.csv", "w83_comparison_vs_A.csv", "w83_by_category.csv", "w83_risk_validation.csv", "w83_train_test_stability.csv", "w83_lead_time_by_supplier.csv"]
check("every output table carries the sales-scale label", "7 of 7", "7 of 7", all("label" in pd.read_csv(OUT / f).columns for f in need))
check("summary, pair and comparison tables carry an assumptions column", "3 of 3", "3 of 3", all("assumptions" in pd.read_csv(OUT / f).columns for f in need[:3]))
check("output tables written (seven)", "7", str(len(outs)), len(outs) == 7)
V = pd.DataFrame(checks); V.insert(0, "Check_ID", range(1, len(V) + 1)); V.to_csv("validation_report_week08_w83.csv", index=False)
print(len(V), "checks,", int((V.Status == "PASS").sum()), "pass"); assert (V.Status == "PASS").all()

31 checks, 31 pass
